<a href="https://colab.research.google.com/github/Shikher-jain/Data_Science/blob/main/ML/cnn_test_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Folders created")

Folders created


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import pandas as pd

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [ ]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear(128*4*4,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )

    def forward(self,x):

        x = self.conv(x)

        x = x.view(x.size(0),-1)

        x = self.fc(x)

        return x

In [ ]:
model = CNN().to(device)

print(model)

CNN(
  (conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=2048, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images,labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted==labels).sum().item()

accuracy = 100*correct/total

print("Test Accuracy:",accuracy)

Test Accuracy: 10.09


In [ ]:
epochs = 30

for epoch in range(epochs):

    running_loss = 0

    for images,labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss:{running_loss/len(train_loader):.4f}")

Epoch 1/30 Loss:1.3646
Epoch 2/30 Loss:0.9144
Epoch 3/30 Loss:0.7163
Epoch 4/30 Loss:0.5730
Epoch 5/30 Loss:0.4534
Epoch 6/30 Loss:0.3370
Epoch 7/30 Loss:0.2480
Epoch 8/30 Loss:0.1672
Epoch 9/30 Loss:0.1308
Epoch 10/30 Loss:0.1058
Epoch 11/30 Loss:0.0852
Epoch 12/30 Loss:0.0914
Epoch 13/30 Loss:0.0741
Epoch 14/30 Loss:0.0702
Epoch 15/30 Loss:0.0723
Epoch 16/30 Loss:0.0672
Epoch 17/30 Loss:0.0579
Epoch 18/30 Loss:0.0565
Epoch 19/30 Loss:0.0564
Epoch 20/30 Loss:0.0593
Epoch 21/30 Loss:0.0582
Epoch 22/30 Loss:0.0467
Epoch 23/30 Loss:0.0614
Epoch 24/30 Loss:0.0421
Epoch 25/30 Loss:0.0576
Epoch 26/30 Loss:0.0576
Epoch 27/30 Loss:0.0449
Epoch 28/30 Loss:0.0465
Epoch 29/30 Loss:0.0444
Epoch 30/30 Loss:0.0485


In [ ]:
torch.save(
    model.state_dict(),
    "checkpoints/cnn_model.pth"
)

print("Model saved")

Model saved


In [ ]:
predictions = []

model.eval()

with torch.no_grad():

    for images,_ in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        predictions.extend(predicted.cpu().numpy())

In [ ]:
df = pd.DataFrame(predictions,columns=["prediction"])

df.to_csv("outputs/predictions.csv",index=False)

print("Predictions saved")

Predictions saved


```
project/
│
├── notebook.ipynb
├── data/
│
├── checkpoints/
│   └── cnn_model.pth
│
├── outputs/
│   └── predictions.csv
```